byte_tracker.py

In [29]:
from trackers.byte_tracker import BYTETracker

import cv2
import torch
import torchvision
import torchvision.transforms as T
import numpy as np
from PIL import Image
from collections import defaultdict
import time
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from types import SimpleNamespace

In [30]:

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [31]:
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# Define the transform (same as during training)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and modify the detection model
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    num_classes=2,
    weights_backbone=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
)
model_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Model.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()


SSD(
  (backbone): SSDLiteFeatureExtractorMobileNet(
    (features): Sequential(
      (0): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (2): Hardswish()
        )
        (1): InvertedResidual(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
              (2): ReLU(inplace=True)
            )
            (1): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
            )
          )
        )
        (2): Invert

Helper Functions

In [32]:
def xyxy_to_xywh(xyxy):
    """Convert bounding box from [x1, y1, x2, y2] to [cx, cy, w, h]."""
    x1, y1, x2, y2 = xyxy[:4]
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    w = x2 - x1
    h = y2 - y1
    return np.array([cx, cy, w, h], dtype=xyxy.dtype)

def predict(image_input, model, device, threshold=0.2, nms_threshold=0.2):
    """
    Predict detections on an image using the SSDLite model.
    
    Args:
        image_input (str or np.ndarray): File path or OpenCV BGR image.
        model: The SSDLite detection model.
        device: Computation device.
        threshold (float): Confidence threshold.
        nms_threshold (float): (Currently unused) IoU threshold.
        
    Returns:
        orig_img (np.ndarray): The original image in BGR format.
        detections (np.ndarray): Detections in [x1, y1, x2, y2, score] format.
    """
    # Convert input to PIL image and copy for display
    if isinstance(image_input, np.ndarray):
        pil_img = Image.fromarray(cv2.cvtColor(image_input, cv2.COLOR_BGR2RGB))
        orig_img = image_input.copy()
    elif isinstance(image_input, str):
        pil_img = Image.open(image_input).convert("RGB")
        orig_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    else:
        raise ValueError("Unsupported image input type.")
    
    # Preprocess and add batch dimension
    img_tensor = transform(pil_img).to(device)
    img_tensor = img_tensor.unsqueeze(0)

    # Run inference and measure time (optional)
    start = cv2.getTickCount()
    with torch.no_grad():
        outputs = model(img_tensor)
    end = cv2.getTickCount()
    inference_time_ms = (end - start) / cv2.getTickFrequency() * 1000.0
    print(f"Inference time: {inference_time_ms:.1f} ms")

    output = outputs[0]
    boxes = output['boxes'].cpu().numpy()   # shape: (N, 4)
    scores = output['scores'].cpu().numpy()   # shape: (N,)

    # Filter out detections below the threshold
    keep = scores >= threshold
    boxes = boxes[keep]
    scores = scores[keep]

    # For demonstration, keep only the highest scoring detection (optional)
    if len(boxes) > 0:
        best_idx = np.argmax(scores)
        boxes = boxes[best_idx:best_idx+1]
        scores = scores[best_idx:best_idx+1]

    if len(boxes) > 0:
        detections = np.hstack((boxes, scores.reshape(-1, 1)))
    else:
        detections = np.empty((0, 5), dtype=np.float32)
    
    return orig_img, detections

In [33]:
def create_dummy_results(dets):
    """
    Given dets (an array of shape (N, 5) with [x1, y1, x2, y2, score]),
    return a dummy results object with attributes:
      - xywh: bounding boxes in [cx, cy, w, h] format
      - conf: confidence scores
      - cls: class labels (assumed to be 0 for all detections)
    """
    if len(dets) == 0:
        return SimpleNamespace(
            xywh=np.empty((0, 4), dtype=np.float32),
            conf=np.empty((0,), dtype=np.float32),
            cls=np.empty((0,), dtype=np.int32)
        )
    boxes_xywh = np.array([xyxy_to_xywh(det[:4]) for det in dets])
    conf = dets[:, 4]
    cls = np.zeros_like(conf, dtype=np.int32)
    return SimpleNamespace(xywh=boxes_xywh, conf=conf, cls=cls)

 Tracker Setup


In [51]:
class DummyArgs:
    track_thresh = 0.2        # Minimum detection confidence for tracking
    track_buffer = 40         # Maximum frames to keep a lost track
    match_thresh = 0.6        # Threshold for matching (e.g., IoU)
    mot20 = False             # MOT20-specific settings flag
    track_low_thresh = 0.1    # Low threshold for detections (second association)
    track_high_thresh = 0.2   # High threshold for detections (first association)
    new_track_thresh = 0.2    # Threshold to initialize a new track
    fuse_score = False        # Whether to fuse detection score

args = DummyArgs()
# Import BYTETracker from your trackers package
from trackers.byte_tracker import BYTETracker
tracker = BYTETracker(args=args, frame_rate=15)

In [53]:
import cv2
import numpy as np
import time
from collections import defaultdict

# Assuming predict, create_dummy_results, and BYTETracker (tracker) are already defined and imported.

video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\TestFile_video.mp4"
video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\Baseline.mp4"
cap = cv2.VideoCapture(video_path)
cv2.namedWindow("Tracking", cv2.WINDOW_NORMAL)

# Dictionary to store track history for drawing tracking lines: track_id -> list of (cx, cy)
track_history = defaultdict(list)
start_time_global = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    elapsed = time.time() - start_time_global
    if elapsed > 60:
        print("Stopping processing: 60-second limit reached.")
        break

    orig_img, detections = predict(frame, model, device, threshold=0.2)
    print("Detections shape:", detections.shape)
    print("Detections:", detections)

    # Create dummy results object from detections
    results = create_dummy_results(detections)
    # Update tracker using the dummy results object (tracker.update expects a results object)
    tracks = tracker.update(results, img=None)
    print("Tracked outputs:")
    print(tracks)

    # Print track IDs and scores to the console (track_id at column 4, score at column 5)
    if tracks.size > 0:
        for track in tracks:
            print("Track ID:", int(track[4]), "Score:", track[5])

    # Annotate the original image with detection boxes (from dummy results)
    def plot_detections(orig_img, dummy_results):
        annotated_img = orig_img.copy()
        for box, score in zip(dummy_results.xywh, dummy_results.conf):
            # Convert xywh back to xyxy for plotting
            cx, cy, w, h = box
            x1 = cx - w / 2
            y1 = cy - h / 2
            x2 = cx + w / 2
            y2 = cy + h / 2
            cv2.rectangle(annotated_img, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
        return annotated_img

    annotated_frame = plot_detections(orig_img, results)
    
    # Overlay track IDs and scores on the frame:
    # Each track is [x1, y1, x2, y2, track_id, score, cls, idx]
    if tracks.size > 0:
        for track in tracks:
            x1, y1, x2, y2, tid, score, cls, idx = track
            # Draw a rectangle (optional) is already drawn in the detections plot
            # Overlay text with both track ID and score
            label = f"ID:{int(tid)}, {score:.2f}"
            cv2.putText(annotated_frame, label, (int(x1), int(y1)-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    cv2.imshow("Tracking", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Exiting display loop.")
        break

cap.release()
cv2.destroyAllWindows()
print("Video processing complete.")


Inference time: 44.7 ms
Detections shape: (1, 5)
Detections: [[430.42117   125.11832   576.2175    373.87344     0.9999361]]
Tracked outputs:
[]
Inference time: 49.1 ms
Detections shape: (1, 5)
Detections: [[433.19302   125.92499   575.1064    375.72467     0.9999014]]
Tracked outputs:
[[430.86282   125.76366   577.1045    375.35443     2.          0.9999014
    0.          0.       ]]
Track ID: 2 Score: 0.9999014
Inference time: 22.0 ms
Detections shape: (1, 5)
Detections: [[431.5706    124.844025  573.9587    374.6537      0.9999169]]
Tracked outputs:
[[429.91373   125.055595  576.17645   374.8149      2.          0.9999169
    0.          0.       ]]
Track ID: 2 Score: 0.9999169
Inference time: 20.6 ms
Detections shape: (1, 5)
Detections: [[432.17935    122.78602    574.56995    375.07153      0.99990416]]
Tracked outputs:
[[429.56943    123.21816    576.8393     374.9587       2.
    0.99990416   0.           0.        ]]
Track ID: 2 Score: 0.99990416
Inference time: 19.8 ms
Detect